# Getting started with battread

The simplest workflow is to read one file and save the standardized result:

```python
import battread

data = battread.read("experiment.csv")
battread.write(data, "standardized.parquet")
```

If you only need an output file, use `battread.convert("experiment.csv", "standardized.parquet")`.
It avoids collecting the full result in RAM. Start with the first three steps
below; the remaining sections are optional.

This notebook is entirely in English and uses generated synthetic data. It
contains no private acquisitions. Run cells from top to bottom in a Python
3.11+ environment with battread installed.

From a repository checkout, install with `python -m pip install .` and install
Jupyter separately with `python -m pip install jupyterlab`. MPR additionally
requires `python -m pip install ".[biologic]"`; Neware binary inputs require
`python -m pip install ".[neware]"`. These commands belong in a terminal, using
the same Python environment as the notebook kernel.


## 1. Prepare a small example

This cell creates two tiny CSV files in a temporary directory so the notebook
runs without downloading data or editing your existing files. In your own
workflow, replace `source` with the path to your measurement file and choose a
permanent output folder.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import battread

workspace = TemporaryDirectory(prefix="battread-example-")
folder = Path(workspace.name)
source = folder / "experiment.csv"
source.write_text(
    "Time (s),Current (mA),Voltage (V)\n10,-2,3.5\n11,3,3.6\n13,-4,3.7\n",
    encoding="utf-8",
)
second_source = folder / "second.csv"
second_source.write_text(
    "Time (s),Current (mA),Voltage (V)\n20,-1,3.8\n21,2,3.9\n",
    encoding="utf-8",
)

## 2. Read the data

`read()` returns a pandas DataFrame with exactly `time_s`, `current_mA`, and
`voltage_V`, all float64. Time starts at zero; measured current keeps its sign.
Here the source times 10, 11, 13 seconds become 0, 1, 3 seconds. Nothing is
sorted, downsampled or silently removed. `read()` holds the complete result in
RAM, so use the streaming section below for large acquisitions.


In [ ]:
data = battread.read(source)
data

## 3. Save the standardized data

The destination extension selects Parquet, CSV or TXT. Parquet preserves the
canonical schema and is recommended for repeated use and large datasets.
CSV uses commas and TXT uses tabs; neither writes a pandas index.
The final file appears only after validation succeeds. An existing destination
raises an error; pass `overwrite=True` only when replacement is intentional.


In [ ]:
saved = battread.write(data, folder / "standardized.parquet")
round_trip = battread.read(saved)
round_trip

## Optional: convert directly, especially for large files

`convert()` reads and writes in chunks instead of assembling a full DataFrame.
For example, with the Bio-Logic extra installed:

```python
battread.convert(
    "large.mpr", "standardized.parquet", chunk_size=10_000,
    columns={"voltage": "Ewe/V"},
)
```

MPR now scans small metadata and reads bounded binary batches instead of loading
the complete measurement module through Galvani. Unsupported layouts or fields
still fail explicitly. Chunk size limits rows per yielded frame, not total
process memory; wide MPR tables can yield smaller frames.

Parquet may make the stored output smaller and avoids parsing the vendor file
again. That is separate from the RAM benefit of streaming. Only the three
canonical columns are retained; retain your source acquisition for its extra
metadata and measurements.


In [ ]:
converted = battread.convert(source, folder / "converted.parquet", chunk_size=2)
battread.read(converted)

## Optional: process one chunk at a time

Consume each frame and release it before moving on. Do not use `list()` or
concatenate every chunk if the purpose is to keep memory low. Exhaust the
iterator to validate the whole source. Closing it after an early stop releases
its file handles. Time validation and reconstruction state span chunk boundaries.


In [ ]:
rows_processed = 0
for chunk in battread.iter_read(source, chunk_size=2):
    rows_processed += len(chunk)
print(f"Processed {rows_processed} synthetic rows")

## Optional: merge files in a chosen order

Read and standardize each input first. `merge()` accepts canonical DataFrames,
not filenames. It appends them in your chosen order and uses an established
adjacent positive time interval at each boundary. If no valid interval exists,
it fails rather than inventing one. This example holds all frames in RAM;
core multi-file streaming merge is not available.


In [ ]:
parts = [battread.read(source), battread.read(second_source)]
combined = battread.merge(parts)
battread.write(combined, folder / "merged.csv")
combined

## Optional: understand column recognition

`inspect()` explains selected, rejected and ambiguous candidates without
standardizing all data rows. It does not certify the complete acquisition.
Recognition uses labels, declared aliases, units and vendor hints; scientific
decisions never rely on fuzzy matching.


In [ ]:
information = battread.inspect(source)
for match in information.columns:
    print(match.source_column, match.quantity, match.state, match.unit)

## Optional: declare columns and units

For unusual exports, name the source columns and their units. Names or zero-based
positions are supported; duplicate labels require positions. Options such as
`sep`, `decimal`, `encoding`, `skiprows`, and `header` declare text structure.
The example below converts minutes to seconds, amperes to milliamperes, and
millivolts to volts.

For Bio-Logic files containing both `Ewe/V` and `<Ewe>/V`, explicitly pass
`columns={"voltage": "Ewe/V"}` if you choose the former. The library does not
assume these columns are equivalent. The desktop app uses that explicit
preference. Neware's verified CSV profile selects **Total Time**, not resetting
step time, and converts its clock strings to elapsed seconds automatically.


In [ ]:
custom_source = folder / "custom.txt"
custom_source.write_text(
    "Elapsed;Measured I;Cell E\n1;-0.002;3500\n2;0.003;3600\n",
    encoding="utf-8",
)
custom_data = battread.read(
    custom_source,
    columns={"time": "Elapsed", "current": "Measured I", "voltage": "Cell E"},
    units={"time": "min", "current": "A", "voltage": "mV"},
    sep=";",
)
custom_data

## Optional: warnings and scientific limits

Missing measurements remain in place and are reported with standard Python
warnings. Backward time, infinity, malformed record structures and scientific
ambiguity raise domain exceptions. A successful conversion is not a substitute
for comparing your data with the original export.

Direct measured current always takes precedence, including missing measured
values. Current reconstruction is used only when current is absent and capacity
semantics are established. Verified Bio-Logic `dq` is preferred in that fallback.
The library does not perform cycle, capacity or energy analysis.


## Optional: parallel processing

There is no built-in parallel batch scheduler. Independent files can be
converted in separate processes, each with its own output path. More workers
increase aggregate memory and disk contention; speed gains require measurement.
Start with two workers and an explicit shared RAM budget rather than using every
CPU core. The desktop app currently processes files sequentially.

Do not split one acquisition into independently normalized chunks: time and
reconstruction state must remain consistent across boundaries. On Windows,
process-worker functions must live in an importable module with a guarded script
entry point; a function defined in a notebook cell is not a portable process-pool
example. See the large-file guide for the planned concurrency benchmark.


## Clean up the synthetic workspace

All files in this example were temporary. This final cell removes only the
notebook-generated temporary directory; it does not touch your acquisitions.


In [ ]:
workspace.cleanup()